In [55]:
import requests

api_key = "344950c3abc0274b486ec96773e0d49a"
ville = "Paris"
url = f"http://api.openweathermap.org/data/2.5/weather?q={ville}&appid={api_key}&units=metric"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    temp_celsius = data["main"]["temp"]
    print("Température à Paris :", temp_celsius, "°C")
else:
    print("Erreur :", response.status_code)

Erreur : 401


In [79]:
pokemon= (input("Pokemon : ").lower().strip())

url = f"https://pokeapi.co/api/v2/pokemon/{pokemon}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    types = [elt['type']['name'] for elt in data['types']]
    print(f"Nom : {data['name']}")
    print(f"Id : {data['id']}")
    print(f"Taille : {data['height']}")
    print(f"Poids : {data['weight']}")
    print(f"Types : {types}")
elif response.status_code == 404:
    print("Désolé, ce Pokémon est introuvable !")
else:
    print("Erreur :", response.status_code)

Nom : bulbasaur
Id : 1
Taille : 7
Poids : 69
Types : ['grass', 'poison']


In [80]:
import requests

# URL où envoyer les données
url = "https://jsonplaceholder.typicode.com/todos"

# 1. Les données que l'on souhaite envoyer (sous forme de dictionnaire Python)
title = input("Quel est le nom de la tâche à ajouter? ")
new_task = {
    "title": title,
    "completed": False,
    "userId": 1
}

# 2. On utilise requests.post() avec le paramètre json=
response = requests.post(url, json=new_task)

# 3. Le code 201 signifie "Created" (Créé avec succès)
if response.status_code == 201:
    donnees_creees = response.json()
    print("✅ Données enregistrées par le serveur !")
    print("Réponse du serveur :", donnees_creees)
else:
    print("Erreur :", response.status_code)

✅ Données enregistrées par le serveur !
Réponse du serveur : {'title': 'Mon premier POST', 'body': "Apprendre les API en Python, c'est super efficace !", 'userId': 1, 'id': 101}


In [99]:
champion = (input("Champion : ").capitalize().strip())

url = "https://ddragon.leagueoflegends.com/cdn/13.24.1/data/fr_FR/champion.json"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    if champion in data['data']:
        role = ", ".join([elt for elt in data['data'][champion]['tags']] )
        print(f"Nom & titre : {data['data'][champion]['id']}, {data['data'][champion]['title']}")
        print(f"Rôle(s) : {role}")
        print(f"Histoire : {data['data'][champion]['blurb']}")
    else:
        print("Désolé, ce champion n'existe pas !")
else:
    print("Erreur :", response.status_code)

Nom & titre : Garen, Force de Demacia
Rôle(s) : Fighter, Tank
Histoire : Garen est un guerrier fier et noble qui fait partie du Détachement hardi. Héritier des Crownguard, la famille chargée de défendre Demacia et ses idéaux, il est apprécié par ses compatriotes et respecté par ses ennemis. Équipé d'une armure résistante à...
Désolé, ce champion est introuvable !


In [100]:
import requests

url = "https://ddragon.leagueoflegends.com/cdn/13.24.1/data/fr_FR/champion.json"

try:
    # 1. On ajoute TOUJOURS un timeout (en secondes)
    response = requests.get(url, timeout=5)

    # 2. On demande à requests de lever une erreur si le statut est 4xx ou 5xx
    response.raise_for_status()

    # Si tout s'est bien passé, on traite les données
    data = response.json()
    print("✅ Connexion réussie et données récupérées !")

except requests.exceptions.ConnectionError:
    print("❌ Erreur : Pas de connexion Internet ou serveur inaccessible.")

except requests.exceptions.Timeout:
    print("⏳ Erreur : Le serveur a mis plus de 5 secondes à répondre.")

except requests.exceptions.HTTPError as err:
    print(f"⚠️ Erreur HTTP (ex: 404 ou 500) : {err}")

except requests.exceptions.RequestException as err:
    # Capturer n'importe quelle autre erreur liée à requests
    print(f"🚨 Une erreur inattendue s'est produite : {err}")

✅ Connexion réussie et données récupérées !


In [ ]:
import requests
import time

def obtenir_donnees_avec_retry(url, max_retries=3, pause=2):
    for tentative in range(1, max_retries + 1):
        try:
            print(f"Tentative {tentative}/{max_retries}...")
            response = requests.get(url, timeout=5)
            response.raise_for_status()

            # Si la requête réussit, on renvoie immédiatement le résultat
            return response.json()

        except requests.exceptions.RequestException as err:
            print(f"⚠️ Échec de la tentative {tentative} : {err}")

            # Si on n'a pas encore atteint la limite, on attend avant de retenter
            if tentative < max_retries:
                print(f"⏳ Pause de {pause} secondes avant le prochain essai...")
                time.sleep(pause)
            else:
                print("❌ Abandon : Nombre maximum de tentatives atteint.")
                return None

# Exemple d'utilisation
data = obtenir_donnees_avec_retry("https://ddragon.leagueoflegends.com/cdn/13.24.1/data/fr_FR/champion.json")

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

# 1. On configure les règles de retry
strategie_retry = Retry(
    total=3,                  # Nombre total de tentatives
    backoff_factor=1,         # Attente exponentielle : 1s, 2s, 4s...
    status_forcelist=[500, 502, 503, 504], # Retenter aussi sur ces erreurs serveur
    raise_on_status=False
)

# 2. On crée une session et on lui attache la stratégie de retry
session = requests.Session()
adapter = HTTPAdapter(max_retries=strategie_retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

# 3. On utilise 'session.get' au lieu de 'requests.get'
url = "https://ddragon.leagueoflegends.com/cdn/13.24.1/data/fr_FR/champion.json"

try:
    # La session retentera automatiquement 3 fois en cas de coupure réseau ou d'erreur 500
    response = session.get(url, timeout=5)
    response.raise_for_status()
    print("✅ Données récupérées avec succès !")

except requests.exceptions.RequestException as err:
    print(f"🚨 Échec final après 3 tentatives : {err}")